# 🤖 Robô Sentinela — Treinando o cérebro (Machine Learning)

Neste notebook você vai:
1. Carregar os dados que o seu grupo gravou com o **sketch 03**;
2. Olhar os dados e entender o que cada *feature* significa;
3. Treinar uma **Árvore de Decisão** que reconhece 4 gestos;
4. Medir se ela funciona (acurácia e matriz de confusão) e discutir *overfitting*;
5. **Converter a árvore em código C** (`modelo.h`) para rodar dentro do Arduino Uno.

| Classe | Gesto |
|---|---|
| 0 | PARADO (mão parada ou ninguém) |
| 1 | APROXIMAR |
| 2 | AFASTAR |
| 3 | ACENAR |

**Por que árvore de decisão?** O Uno tem só 2 KB de RAM. Uma árvore vira uma sequência de `if/else` — roda em microssegundos e quase não ocupa memória. Redes neurais também são possíveis em microcontroladores (TinyML), mas exigem placas maiores.

➡️ Execute as células em ordem (Shift+Enter).

In [ ]:
import io, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, classification_report

FEATURES = ["media", "desvio", "minimo", "maximo", "inclinacao", "trocas"]
NOMES_CLASSES = ["PARADO", "APROXIMAR", "AFASTAR", "ACENAR"]
print("Bibliotecas carregadas ✔")

## 1. Carregar os dados

Envie o(s) arquivo(s) `dados_grupoX.csv` (o texto copiado do Serial Monitor).
Pode enviar mais de um arquivo — por exemplo, os dados de **todos os grupos** juntos.

A função abaixo é tolerante: ignora comentários (`#`), cabeçalhos repetidos e o horário que o Serial Monitor coloca quando o *timestamp* está ligado.

In [ ]:
def ler_csv_serial(texto, origem):
    linhas = []
    for bruta in texto.splitlines():
        linha = bruta.split("->")[-1].strip()      # remove o timestamp "12:00:01.123 -> "
        if not linha or linha.startswith("#"):
            continue
        partes = linha.split(",")
        if len(partes) != 7:
            continue
        try:
            valores = [float(p) for p in partes]
        except ValueError:
            continue                                # cabeçalho ou lixo
        linhas.append(valores)
    d = pd.DataFrame(linhas, columns=FEATURES + ["classe"])
    d["classe"] = d["classe"].astype(int)
    d["origem"] = origem
    return d

try:
    from google.colab import files               # no Colab: abre a janela de envio
    enviados = files.upload()
    conteudos = {nome: dados.decode("utf-8", errors="ignore") for nome, dados in enviados.items()}
except ImportError:                              # fora do Colab: lê os .csv da pasta
    conteudos = {nome: open(nome, encoding="utf-8", errors="ignore").read()
                 for nome in sorted(glob.glob("dados_*.csv"))}

df = pd.concat([ler_csv_serial(txt, nome) for nome, txt in conteudos.items()], ignore_index=True)
print(f"{len(df)} amostras carregadas de {df['origem'].nunique()} arquivo(s)")
print(df.groupby(["origem", "classe"]).size().unstack(fill_value=0))
df.head()

## 2. Explorar os dados

Antes de treinar qualquer coisa, **olhe os dados**. Pergunta para o grupo: *qual feature parece separar melhor APROXIMAR de AFASTAR? E ACENAR do resto?*

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))
cores = ["#7f7f7f", "#d62728", "#1f77b4", "#2ca02c"]
for c in range(4):
    sub = df[df["classe"] == c]
    eixos[0].scatter(sub["inclinacao"], sub["trocas"] + np.random.uniform(-0.15, 0.15, len(sub)),
                     c=cores[c], label=NOMES_CLASSES[c], alpha=0.7)
    eixos[1].scatter(sub["media"], sub["desvio"], c=cores[c], label=NOMES_CLASSES[c], alpha=0.7)
eixos[0].set_xlabel("inclinação (cm/s)"); eixos[0].set_ylabel("trocas de sentido")
eixos[1].set_xlabel("média (cm)");        eixos[1].set_ylabel("desvio padrão (cm)")
for e in eixos: e.legend(); e.grid(alpha=0.3)
plt.tight_layout(); plt.show()

df.groupby("classe")[FEATURES].mean().rename(index=dict(enumerate(NOMES_CLASSES))).round(1)

## 3. Treinar a árvore de decisão

Separamos **75% para treino** e **25% para teste**. O modelo nunca vê os dados de teste durante o treino — é assim que sabemos se ele *aprendeu* ou só *decorou*.

`max_depth` = quantas perguntas (no máximo) a árvore pode fazer em sequência. Comece com 4.

In [ ]:
PROFUNDIDADE = 4   # experimente 1, 2, 3, 4, 6, 10...

X, y = df[FEATURES].values, df["classe"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

arvore = DecisionTreeClassifier(max_depth=PROFUNDIDADE, min_samples_leaf=2, random_state=42)
arvore.fit(X_tr, y_tr)

print(f"Acurácia no TREINO: {accuracy_score(y_tr, arvore.predict(X_tr)):.1%}")
print(f"Acurácia no TESTE : {accuracy_score(y_te, arvore.predict(X_te)):.1%}")

## 4. Onde o modelo erra? (matriz de confusão)

Cada linha é o gesto **verdadeiro**; cada coluna é o que o modelo **respondeu**. A diagonal são os acertos.

In [ ]:
ConfusionMatrixDisplay.from_estimator(arvore, X_te, y_te, display_labels=NOMES_CLASSES,
                                      cmap="Blues", colorbar=False)
plt.title("Matriz de confusão — dados de teste"); plt.show()
print(classification_report(y_te, arvore.predict(X_te), target_names=NOMES_CLASSES, zero_division=0))

## 5. Abrindo a "caixa": como a árvore decide

Diferente de uma rede neural, dá para ler a árvore. Compare com as regras que vocês tentaram escrever à mão no sketch 02.

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(arvore, feature_names=FEATURES, class_names=NOMES_CLASSES, filled=True, rounded=True, fontsize=9)
plt.show()

pd.Series(arvore.feature_importances_, index=FEATURES).sort_values().plot.barh(title="Importância das features")
plt.show()

## 6. Experimento: overfitting

Árvore rasa demais → não aprende (*underfitting*). Funda demais → decora o treino e erra em dado novo (*overfitting*).
Usamos validação cruzada (5 divisões) para escolher a profundidade com mais confiança.

In [ ]:
profs = range(1, 11)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
treino, validacao = [], []
for p in profs:
    m = DecisionTreeClassifier(max_depth=p, min_samples_leaf=2, random_state=42)
    validacao.append(cross_val_score(m, X, y, cv=cv).mean())
    treino.append(m.fit(X, y).score(X, y))
plt.plot(profs, treino, "o-", label="treino")
plt.plot(profs, validacao, "s-", label="validação cruzada")
plt.xlabel("max_depth"); plt.ylabel("acurácia"); plt.ylim(0, 1.05); plt.grid(alpha=0.3); plt.legend()
plt.title("Profundidade da árvore x acurácia"); plt.show()
melhor = list(profs)[int(np.argmax(validacao))]
print(f"Melhor profundidade pela validação cruzada: {melhor}")

## 7. (Desafio) O modelo funciona com OUTRAS pessoas?

Só roda se vocês enviaram arquivos de mais de um grupo. Treinamos com todos os grupos menos um e testamos no que ficou de fora.

In [ ]:
origens = df["origem"].unique()
if len(origens) < 2:
    print("Envie os CSVs de pelo menos 2 grupos para este experimento.")
else:
    for fora in origens:
        tr, te = df[df["origem"] != fora], df[df["origem"] == fora]
        m = DecisionTreeClassifier(max_depth=PROFUNDIDADE, min_samples_leaf=2, random_state=42)
        m.fit(tr[FEATURES], tr["classe"])
        print(f"Testando em {fora:25s}: {m.score(te[FEATURES], te['classe']):.1%}")

## 8. Exportar o cérebro para o Arduino 🚀

Treinamos de novo com **todos** os dados e convertemos a árvore em uma função C:

```c
int classificar(const float *f);   // f[0..5] = as 6 features, devolve a classe 0..3
```

O arquivo `modelo.h` será baixado. Coloque-o na pasta do **sketch 04** (substituindo o de exemplo) e faça o upload.

In [ ]:
def exportar_arvore_c(modelo, nomes, classes, caminho="modelo.h"):
    t = modelo.tree_
    linhas = []
    def no(i, nivel):
        esp = "  " * nivel
        if t.feature[i] != _tree.TREE_UNDEFINED:
            k = t.feature[i]
            linhas.append(f"{esp}if (f[{k}] <= {t.threshold[i]:.4f}f) {{   // {nomes[k]}")
            no(t.children_left[i], nivel + 1)
            linhas.append(f"{esp}}} else {{")
            no(t.children_right[i], nivel + 1)
            linhas.append(f"{esp}}}")
        else:
            c = int(modelo.classes_[np.argmax(t.value[i][0])])
            linhas.append(f"{esp}return {c};   // {classes[c]}")
    no(0, 1)
    cabecalho = [
        "// modelo.h — gerado automaticamente no Colab (Robô Sentinela)",
        f"// Árvore de decisão: profundidade {modelo.get_depth()}, {modelo.get_n_leaves()} folhas",
        "// " + "  ".join(f"f[{j}]={n}" for j, n in enumerate(nomes)),
        "#pragma once",
        "",
        "int classificar(const float *f) {",
    ]
    codigo = "\n".join(cabecalho + linhas + ["}", ""])
    with open(caminho, "w") as arq:
        arq.write(codigo)
    return codigo

modelo_final = DecisionTreeClassifier(max_depth=PROFUNDIDADE, min_samples_leaf=2, random_state=42).fit(X, y)
print(exportar_arvore_c(modelo_final, FEATURES, NOMES_CLASSES))

try:
    from google.colab import files
    files.download("modelo.h")
except ImportError:
    print("modelo.h salvo na pasta atual.")